In [20]:
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import json
import os

In [60]:
class BaloonDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms

        annotation_file = f'{root}/via_region_data.json'
        with open(annotation_file, 'r') as f:
            self.annotations = json.load(f)
        # self.image_ids = [entry['filename'] for entry in self.annotations.values()]
        self.image_ids = list(self.annotations.keys())

    def __getitem__(self, idx):
        img_key = self.image_ids[idx]
        print(img_key)
        ann = self.annotations[img_key]
        print(ann)
        img_name = ann["filename"]
        print(img_name)
        img_path = os.path.join(self.root, img_name)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Image {img_path} not found.")
        img = Image.open(img_path).convert("RGB")

        ann = self.annotations[img_key]
        boxes = []

        for region in ann["regions"].values():
            shape = region["shape_attributes"]
            xmin = shape["x"]
            ymin = shape["y"]
            xmax = xmin + shape["width"]
            ymax = ymin + shape["height"]
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)  # All balloons
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_ids)

In [61]:
ballon = BaloonDataset(root='balloon/train',
    transforms=transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]))
len(ballon)

61

In [62]:
ballon[9]

15290896925_884ab33fd3_k.jpg1287786
{'fileref': '', 'size': 1287786, 'filename': '15290896925_884ab33fd3_k.jpg', 'base64_img_data': '', 'file_attributes': {}, 'regions': {'0': {'shape_attributes': {'name': 'polygon', 'all_points_x': [1388, 1406, 1413, 1414, 1413, 1402, 1389, 1370, 1349, 1325, 1309, 1297, 1285, 1272, 1251, 1230, 1217, 1206, 1211, 1219, 1229, 1247, 1275, 1300, 1326, 1347, 1369, 1388], 'all_points_y': [60, 83, 102, 125, 152, 177, 194, 210, 225, 233, 235, 236, 222, 208, 198, 195, 194, 193, 168, 130, 106, 77, 59, 41, 42, 42, 50, 60]}, 'region_attributes': {}}, '1': {'shape_attributes': {'name': 'polygon', 'all_points_x': [1310, 1306, 1295, 1280, 1268, 1243, 1224, 1205, 1187, 1167, 1154, 1141, 1130, 1128, 1126, 1122, 1123, 1123, 1123, 1122, 1123, 1128, 1131, 1138, 1148, 1156, 1172, 1185, 1201, 1218, 1236, 1255, 1276, 1290, 1301, 1306, 1310, 1310], 'all_points_y': [284, 305, 327, 344, 355, 368, 373, 375, 375, 370, 366, 361, 355, 350, 353, 349, 345, 334, 314, 297, 279, 264, 25

KeyError: 'x'